# LSTM v2
# 예측 범위 16-> 1 => 16 -> 4

# Import

In [ ]:
import os
import sys

import numpy as  np
# from lwm_model import lwm  # 클래스 import
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader, Subset
import torch.nn as nn   
import time

from PIL import Image
import numpy as np

from deepverse import ParameterManager
from deepverse.scenario import ScenarioManager
from deepverse import Dataset

from deepverse.visualizers import ImageVisualizer, LidarVisualizer

# Setting 
### CUDA 몇번 사용하는지 잘 확인하기

In [ ]:
# Scenes 1000
## Subcarriers 64

scenarios_name = "DT31"
config_path = f"scenarios/{scenarios_name}/param/config.m"
param_manager = ParameterManager(config_path)

params = param_manager.get_params()

param_manager.params["scenes"] =list(range(100))
param_manager.params["comm"]["OFDM"]["selected_subcarriers"] = list(range(64))

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")


현재 사용 중인 장치: cuda:1


# Generate a dataset

In [ ]:
# scenes / subcarriers는 그대로
param_manager.params["radar"]["enable"] = False

# lidar가 dict로 존재하면 disable
if isinstance(param_manager.params.get("lidar", None), dict):
    param_manager.params["lidar"]["enable"] = False

# position이 dict로 존재하면 disable
if isinstance(param_manager.params.get("position", None), dict):
    param_manager.params["position"]["enable"] = False

# Radar, LiDAR, Position 사용하지 않으므로 False 
dataset = Dataset(param_manager)

Generating camera dataset: ⏳ In progress
Generating camera dataset: ✅ Completed (0.00s)
Generating LiDAR dataset: ⏳ In progress
Generating LiDAR dataset: ✅ Completed (0.00s)
Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (0.88s)


# Prerocessing

#### Channel

In [ ]:
def get_coeffs_from_frame(frame, ue_idx=0):
    ue_obj = frame["ue"]

    # 케이스1) list/tuple이면 ue_idx로 선택
    if isinstance(ue_obj, (list, tuple)):
        ch_obj = ue_obj[ue_idx]
    else:
        # 케이스2) 단일 OFDMChannel이면 그대로 사용
        ch_obj = ue_obj

    # coeffs는 dict key가 아니라 attribute일 확률이 매우 큼
    if hasattr(ch_obj, "coeffs"):
        return ch_obj.coeffs

    # 혹시 dict라면 마지막 보험
    if isinstance(ch_obj, dict) and "coeffs" in ch_obj:
        return ch_obj["coeffs"]

    raise TypeError(f"Cannot get coeffs. ue type={type(ue_obj)}, ch type={type(ch_obj)}")


In [ ]:
def get_train_min_max_realimag(frames, train_idx, us_idx=0):

    rmin, rmax =  float('inf'), float('-inf')
    imin, imax =  float('inf'), float('-inf')

    print("Calculating min/max over training set...")

    for t  in train_idx:
        frame  = frames[t]
        cooeffs  = get_coeffs_from_frame(frame, us_idx)  # (N_subcarriers, )

        rmin = min(rmin, float(cooeffs.real.min()))
        rmax = max(rmax, float(cooeffs.real.max()))
        imin = min(imin, float(cooeffs.imag.min()))
        imax = max(imax, float(cooeffs.imag.max()))

    print(f"Done. rmin={rmin}, rmax={rmax}, imin={imin}, imax={imax}")
    return (rmin, rmax), (imin, imax)

In [ ]:
def preprocess_channel_coeffs_minmax(coeffs_np, r_min, r_max, i_min, i_max, device=device, eps=1e-16):
    # Convert Numpy to Tensor
    coeffs = torch.from_numpy(coeffs_np).to(torch.complex64)
    
    
    r = coeffs.real
    i = coeffs.imag
    
    # Min-Max Scaling [0, 1]
    # Add eps to denominator to prevent division by zero
    r_scaled = (r - r_min) / max(r_max - r_min, eps)
    i_scaled = (i - i_min) / max(i_max - i_min, eps)
    
    # Concat (Maintains shape like (..., 2*subcarriers))
    H = torch.cat([r_scaled, i_scaled], dim=-1).to(device, dtype=torch.float32)
    return H

#### image

# Dataset 구현

In [ ]:
def flatten_comm_frames(comm):
    frames = []
    for row in comm.data:
        for d in row:
            frames.append(d)
    return frames

class ChannelNextStepDatasetGPU(TorchDataset):
    def __init__(self, comm_frames, ue_idx=0, past_len=16, pre_len=4, device=device,
                 r_min=0.0, r_max=1.0, i_min=0.0, i_max=1.0):
        self.comm_frames = comm_frames
        self.ue_idx = ue_idx
        self.past_len = past_len
        # ✅ pre_len 추가 
        self.pre_len = pre_len 
        self.device = device

        self.r_min, self.r_max = r_min, r_max
        self.i_min, self.i_max = i_min, i_max

        self.N = len(self.comm_frames)
        self.valid_start = past_len - 1
        # ✅ t + pre_len 까지 필요
        self.valid_end = self.N - self.pre_len - 1

    def __len__(self):
        return self.valid_end - self.valid_start + 1

    def __getitem__(self, idx):
        t = self.valid_start + idx

        # 1) Channel Past: (T, 2048)
        ch_list = []
        for k in range(t - self.past_len + 1, t + 1):
            coeffs_np = get_coeffs_from_frame(self.comm_frames[k], ue_idx=self.ue_idx)
            h = preprocess_channel_coeffs_minmax(
                coeffs_np,
                self.r_min, self.r_max, self.i_min, self.i_max,
                device=self.device
            ).reshape(-1)  # (2048,)
            ch_list.append(h)
        channel_past = torch.stack(ch_list, dim=0)  # (past_len, 2048)

        # 2) Target Next 4: flatten -> (4*2048,)
        # target이 1이 아닌 4인 점 주의 
        tgt_list = []
        for j in range(1, self.pre_len + 1):
            coeffs_np_next = get_coeffs_from_frame(self.comm_frames[t + j], ue_idx=self.ue_idx)
            hj = preprocess_channel_coeffs_minmax(
                coeffs_np_next,
                self.r_min, self.r_max, self.i_min, self.i_max,
                device=self.device
            ).reshape(-1)  # (2048,)
            tgt_list.append(hj)

        target = torch.stack(tgt_list, dim=0)  # (pre_len, 2048)

        return channel_past, target

# DataLoader 구현

In [ ]:
comm_frames = flatten_comm_frames(dataset.comm_dataset)

ds = ChannelNextStepDatasetGPU(
    comm_frames=comm_frames,
    ue_idx=0,
    past_len=16,
    pre_len=4,
    device=device
)

loader = DataLoader(
    ds,
    batch_size=8,
    shuffle=True,
    num_workers=0,     
    pin_memory=False   
)
ch, y = next(iter(loader))
print(ch.shape, y.shape)
print(ch.device, y.device)


torch.Size([8, 16, 2048]) torch.Size([8, 2048])
cuda:1 cuda:1


# Fine-tuning

In [ ]:
class LSTMForecaster(nn.Module):
    """
    Input : ch   (B,T,F_in)
    Output: yhat (B,pre_len,F_step)
    """
    def __init__(
        self,
        F_in: int,
        pre_len: int,
        F_step: int,
        hidden: int = 256,
        num_layers: int = 2,
        dropout: float = 0.1,
        pool: str = "last",  # "last" or "mean"
    ):
        super().__init__()
        self.pool = pool
        self.pre_len = pre_len
        self.F_step = F_step

        self.lstm = nn.LSTM(
            input_size=F_in,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0),
        )

        # ✅ pre_len * F_step 만큼 한 번에 뽑고 reshape
        self.head = nn.Linear(hidden, pre_len * F_step)

    def forward(self, ch):
        B = ch.size(0)

        out, _ = self.lstm(ch)  # (B,T,hidden)

        if self.pool == "last":
            z = out[:, -1, :]     # (B,hidden)
        elif self.pool == "mean":
            z = out.mean(dim=1)   # (B,hidden)
        else:
            raise ValueError(f"Unknown pool={self.pool}")

        yhat_flat = self.head(z)  # (B, pre_len*F_step)
        return yhat_flat.view(B, self.pre_len, self.F_step)  # ✅ (B,pre_len,F_step)

# NMSE(dB)

In [ ]:
@torch.no_grad()
def nmse_db(yhat: torch.Tensor, y: torch.Tensor, eps: float = 1e-16) -> torch.Tensor:
    # yhat, y: (B, pre_len, F_step)
    num = torch.sum((yhat - y) ** 2).clamp_min(eps)  # batch 전체 합
    den = torch.sum(y ** 2).clamp_min(eps)
    nmse = num / den
    return 10.0 * torch.log10(nmse)

# Train/val Split

In [ ]:
# 겹치는 부분이 있는지 확인하는 함수
# Target이 1일 때만 아래 코드 작성
# def used_frames_for_ds_indices(ds, ds_indices):
#     used = set()
#     for i in ds_indices:
#         t = ds.valid_start + i
#         # inputs use [t-T+1 .. t], target uses (t+1)
#         used.update(range(t - T + 1, t + 2))
#     return used

def used_frames_for_ds_indices(ds, ds_indices):
    used = set()
    T = ds.past_len
    P = ds.pre_len
    for i in ds_indices:
        t = ds.valid_start + i
        used.update(range(t - T + 1, t + P + 1))  # [t-15 .. t+4]
    return used

In [ ]:
n_total = len(ds)
T = ds.past_len  # = past_len
P = ds.pre_len
# ---- choose k so that train=3k, val=k and NO frame-overlap is possible ----
k_max = (n_total - (T + P - 1)) // 4
if k_max <= 0:
    raise ValueError(
        f"Not enough data for strict non-overlap 3:1. "
        f"n_total={n_total}, past_len(T)={T}. "
        f"Try increasing scenes or decreasing past_len."
    )

k = k_max               # use as much data as possible under constraints
n_val = k
n_train = 3 * k

train_idx = list(range(0, n_train))
val_idx   = list(range(n_total - n_val, n_total))

print("n_total:", n_total, "T:", T)
print("train samples:", len(train_idx), "val samples:", len(val_idx), "ratio:", len(train_idx)/len(val_idx))

# ---- min/max ONLY from train (same as your original policy) ----

# (4,2048) 이 되므로 아래 min/max 계산도 바꿔서
# ---- min/max ONLY from train, but using ALL frames used by train samples ----
train_used_frames = sorted(used_frames_for_ds_indices(ds, train_idx))  # <-- 핵심

(real_min, real_max), (imag_min, imag_max) = get_train_min_max_realimag(
    comm_frames, train_used_frames, us_idx=0
)

ds.r_min = real_min
ds.r_max = real_max
ds.i_min = imag_min
ds.i_max = imag_max

print("Dataset statistical values set in the dataset (from all train-used frames).")

train_ds = Subset(ds, train_idx)
val_ds   = Subset(ds, val_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)

F_in = 2048


overlap = used_frames_for_ds_indices(ds, train_idx).intersection(
          used_frames_for_ds_indices(ds, val_idx))
print("frame-overlap count:", len(overlap))
assert len(overlap) == 0, "Leakage detected: train/val share frames!"

# Verify (channel-only)
ch, y = next(iter(train_loader))
assert y.ndim == 3, f"Expected (B, pre_len, F_step), got {y.shape}"

pre_len = y.shape[1]
F_step  = y.shape[2]
F_out_total = pre_len * F_step

print("\n=== Data Check ===")
print(f"y stats | min: {y.min().item():.4f}, max: {y.max().item():.4f}")
print("If scaling worked correctly, values should be within [0, 1].")

# (선택) ch도 같이 범위 확인하고 싶으면
print(f"ch stats | min: {ch.min().item():.4f}, max: {ch.max().item():.4f}")

n_total: 85 T: 16
train samples: 51 val samples: 17 ratio: 3.0
Calculating min/max over training set...
Done. rmin=-1.0646139639174899e-06, rmax=1.0917989155343993e-06, imin=-1.0719516685941772e-06, imax=1.0643868111374237e-06
Dataset statistical values set in the dataset.
frame-overlap count: 0

=== Data Check ===
y stats | min: 0.0000, max: 1.0000
If scaling worked correctly, values should be within [0, 1].
ch stats | min: -0.2541, max: 1.2434


# Train/Eval 함수

In [ ]:
# batch check
ch, y = next(iter(train_loader))
print(ch.shape, y.shape)

def train_one_epoch(model, loader, optimizer, device, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    total_nmse = 0.0
    n = 0

    for ch, y in loader:
        ch = ch.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        yhat = model(ch)                
        loss = F.mse_loss(yhat, y)
        loss.backward()

        if grad_clip and grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        total_loss += loss.item()
        total_nmse += nmse_db(yhat.detach(), y).item()
        n += 1

    return total_loss / max(n, 1), total_nmse / max(n, 1)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_nmse = 0.0
    n = 0

    for ch, y in loader:
        ch = ch.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        yhat = model(ch)                 
        loss = F.mse_loss(yhat, y)

        total_loss += loss.item()
        total_nmse += nmse_db(yhat, y).item()
        n += 1

    return total_loss / max(n, 1), total_nmse / max(n, 1)

torch.Size([32, 16, 2048]) torch.Size([32, 2048])


# 모델 생성 및 확인

In [ ]:
# 배치 하나로 F_in/F_out 자동 확정
ch, y = next(iter(train_loader))
F_in    = ch.shape[-1]      # 2048
pre_len = y.shape[1]        # 4
F_step  = y.shape[2]        # 2048
F_out_total = pre_len * F_step   # 8192

print("Detected:",
      "F_in=", F_in,
      "pre_len=", pre_len,
      "F_step=", F_step,
      "F_out_total=", F_out_total)
print("Batch devices:", ch.device, y.device)
model = LSTMForecaster(
    F_in=F_in,
    pre_len=pre_len,
    F_step=F_step,
    hidden=256,
    num_layers=2,
    dropout=0.1,
    pool="last",
).to(device)

# sanity forward
model.eval()
with torch.no_grad():
    # ds가 이미 cuda 텐서 반환이면 아래 .to(device) 생략 가능
    yhat = model(ch.to(device))
print("yhat:", yhat.shape, "y:", y.shape)



Detected: F_in= 2048 F_out= 2048
Batch devices: cuda:1 cuda:1
yhat: torch.Size([32, 2048]) y: torch.Size([32, 2048])


# Optimizer/Scheduler 설정

In [ ]:
# requires_grad=True인 파라미터만 학습
trainable_params = [p for p in model.parameters() if p.requires_grad]
print("trainable params:", sum(p.numel() for p in trainable_params))

optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

# (선택) cosine scheduler
epochs = 500
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)



trainable params: 3414016


# 학습 루프 및 checkpoint 저장

In [ ]:
import time
import os
import torch

best_val = float("inf")
best_val_nmse = None
best_epoch = None

t_train0 = time.time()

log_path = "lstm_scene100_v1.txt"

with open(log_path, "w", encoding="utf-8") as f:
    for epoch in range(1, epochs + 1):
        t0 = time.time()

        tr_loss, tr_nmse = train_one_epoch(
            model, train_loader, optimizer, device=device, grad_clip=1.0
        )
        va_loss, va_nmse = evaluate(model, val_loader, device=device)

        scheduler.step()
        dt = time.time() - t0

        line = (
            f"[{epoch:02d}/{epochs}] "
            f"train loss={tr_loss:.6f}, nmse(dB)={tr_nmse:.4f} | "
            f"val loss={va_loss:.6f}, nmse(dB)={va_nmse:.4f} | "
            f"{dt:.1f}s"
        )
        print(line)
        f.write(line + "\n")
        f.flush()

        if va_loss < best_val:
            best_val = va_loss
            best_val_nmse = va_nmse
            best_epoch = epoch

            ckpt_path = "best_lstm_scene100_v1.pt"
            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),

                    "F_in": F_in,
                    "pre_len": pre_len,
                    "F_step": F_step,
                    "F_out_total": pre_len * F_step,

                    "best_val_loss": best_val,
                    "best_val_nmse_db": best_val_nmse,
                },
                ckpt_path,
            )

            save_line = (
                f"  ↳ saved {ckpt_path} | best@{best_epoch}: "
                f"val loss={best_val:.6f}, val nmse(dB)={best_val_nmse:.4f}"
            )
            print(save_line)
            f.write(save_line + "\n")
            f.flush()

total_sec = time.time() - t_train0
h = int(total_sec // 3600)
m = int((total_sec % 3600) // 60)
s = total_sec % 60

summary1 = "\n=== Training Summary ==="
summary2 = f"Total time: {total_sec:.1f}s ({h}h {m}m {s:.1f}s)"
summary3 = (
    f"Best epoch: {best_epoch} | best val loss={best_val:.6f}, "
    f"best val nmse(dB)={best_val_nmse:.4f}"
)

print(summary1)
print(summary2)
print(summary3)

with open(log_path, "a", encoding="utf-8") as f:
    f.write(summary1 + "\n")
    f.write(summary2 + "\n")
    f.write(summary3 + "\n")

print("Log saved to:", os.path.abspath(log_path))

[01/500] train loss=0.294275, nmse(dB)=0.0196 | val loss=0.252619, nmse(dB)=-0.0916 | 0.2s
  ↳ saved bestc_channel_only_finetune.pt | best@1: val loss=0.252619, val nmse(dB)=-0.0916
[02/500] train loss=0.284950, nmse(dB)=-0.0999 | val loss=0.244687, nmse(dB)=-0.2301 | 0.1s
  ↳ saved bestc_channel_only_finetune.pt | best@2: val loss=0.244687, val nmse(dB)=-0.2301
[03/500] train loss=0.277387, nmse(dB)=-0.2200 | val loss=0.235734, nmse(dB)=-0.3920 | 0.1s
  ↳ saved bestc_channel_only_finetune.pt | best@3: val loss=0.235734, val nmse(dB)=-0.3920
[04/500] train loss=0.265277, nmse(dB)=-0.3651 | val loss=0.225125, nmse(dB)=-0.5920 | 0.1s
  ↳ saved bestc_channel_only_finetune.pt | best@4: val loss=0.225125, val nmse(dB)=-0.5920
[05/500] train loss=0.255703, nmse(dB)=-0.5501 | val loss=0.212725, nmse(dB)=-0.8381 | 0.1s
  ↳ saved bestc_channel_only_finetune.pt | best@5: val loss=0.212725, val nmse(dB)=-0.8381
[06/500] train loss=0.242898, nmse(dB)=-0.7634 | val loss=0.198712, nmse(dB)=-1.1340 |

# 런타임 체크

In [ ]:
ch, y = next(iter(train_loader))
print("y abs mean:", y.abs().mean().item())
print("y abs max :", y.abs().max().item())
print("y power   :", (y**2).mean().item())

with torch.no_grad():
    yhat = model(ch.to(device))   
print("yhat abs mean:", yhat.abs().mean().item())
print("yhat abs max :", yhat.abs().max().item())
print("yhat power   :", (yhat**2).mean().item())

y abs mean: 0.49680155515670776
y abs max : 1.0
y power   : 0.29511117935180664
yhat abs mean: 0.49937790632247925
yhat abs max : 1.0309604406356812
yhat power   : 0.2561206817626953


In [ ]:
y.shape

torch.Size([32, 2048])

In [ ]:
ch.shape

torch.Size([32, 16, 2048])

In [ ]:
y

tensor([[0.5352, 0.5520, 0.5665, 0.5792, 0.5914, 0.6044, 0.6190, 0.6354, 0.6526,
         0.6690],
        [0.9370, 0.9367, 0.9338, 0.9288, 0.9222, 0.9149, 0.9068, 0.8974, 0.8859,
         0.8714],
        [0.1570, 0.1559, 0.1549, 0.1553, 0.1578, 0.1624, 0.1687, 0.1759, 0.1833,
         0.1911],
        [0.1215, 0.1107, 0.1027, 0.0966, 0.0915, 0.0864, 0.0810, 0.0755, 0.0709,
         0.0681],
        [0.7071, 0.7267, 0.7437, 0.7581, 0.7704, 0.7818, 0.7933, 0.8053, 0.8177,
         0.8296],
        [0.4669, 0.4773, 0.4875, 0.4993, 0.5141, 0.5321, 0.5526, 0.5738, 0.5938,
         0.6111],
        [0.6921, 0.6864, 0.6767, 0.6626, 0.6449, 0.6252, 0.6056, 0.5878, 0.5729,
         0.5609],
        [0.3030, 0.2955, 0.2899, 0.2847, 0.2787, 0.2710, 0.2619, 0.2519, 0.2422,
         0.2337],
        [0.3408, 0.3589, 0.3811, 0.4058, 0.4306, 0.4531, 0.4719, 0.4865, 0.4978,
         0.5076],
        [0.4504, 0.4602, 0.4715, 0.4856, 0.5034, 0.5244, 0.5470, 0.5693, 0.5893,
         0.6058]], device='c

In [ ]:
yhat

tensor([[0.5693, 0.5512, 0.6085, 0.6257, 0.6239, 0.6050, 0.6008, 0.6040, 0.6270,
         0.5525],
        [0.5422, 0.5546, 0.6376, 0.6435, 0.6420, 0.6333, 0.5448, 0.5250, 0.5519,
         0.5217],
        [0.5665, 0.5935, 0.6417, 0.5889, 0.6679, 0.6315, 0.6241, 0.6522, 0.6280,
         0.5631],
        [0.3962, 0.3751, 0.4180, 0.3361, 0.3698, 0.3214, 0.2135, 0.1952, 0.2300,
         0.3230],
        [0.7148, 0.7834, 0.8037, 0.8573, 0.8253, 0.8665, 0.9184, 0.9386, 0.9786,
         0.8207],
        [0.5462, 0.5478, 0.5804, 0.6099, 0.6020, 0.5746, 0.5621, 0.5898, 0.5931,
         0.5570],
        [0.5694, 0.5553, 0.5585, 0.5422, 0.5536, 0.5487, 0.5615, 0.5531, 0.5415,
         0.5688],
        [0.5843, 0.5460, 0.5202, 0.5167, 0.5050, 0.5240, 0.5364, 0.5313, 0.5314,
         0.5760],
        [0.5672, 0.5690, 0.5545, 0.5485, 0.5350, 0.5556, 0.5720, 0.5596, 0.5454,
         0.5612],
        [0.5659, 0.5767, 0.5400, 0.5466, 0.5456, 0.5394, 0.5598, 0.5461, 0.5345,
         0.5518]], device='c

# 데이터 입력 형태

In [ ]:
print("=== dataset sizes ===")
print("N(comm_frames):", len(comm_frames))
print("past_len      :", ds.past_len)
print("pre_len       :", ds.pre_len)
print("len(ds)       :", len(ds))
print("len(train_ds) :", len(train_ds))
print("len(val_ds)   :", len(val_ds))
print("len(train_loader):", len(train_loader))
print("len(val_loader)  :", len(val_loader))

print("\n=== one batch shapes ===")
ch, y = next(iter(train_loader))

print("ch :", tuple(ch.shape), " -> (B, T, F_in)")
print("y  :", tuple(y.shape),  " -> (B, pre_len, F_step)")

with torch.no_grad():
    yhat = model(ch.to(device))

print("yhat:", tuple(yhat.shape), " -> (B, pre_len, F_step)")

# ✅ sanity: shape 일치 확인
assert yhat.shape == y.shape, f"shape mismatch: yhat={yhat.shape}, y={y.shape}"

B, P, F = yhat.shape
print("this forward predicted samples:", B, "(=B)")
print("each sample predicts steps    :", P, "(=pre_len)")
print("each step predicts elements   :", F, "(=F_step)")

# (선택) 첫 샘플의 1-step 일부 값 확인
print("\n=== sample preview ===")
print("y[0,0,:10]   :", y[0,0,:10].detach().cpu())
print("yhat[0,0,:10]:", yhat[0,0,:10].detach().cpu())

=== dataset sizes ===
N(comm_frames): 101
past_len      : 16
len(ds)       : 85
len(train_ds) : 51
len(val_ds)   : 17
len(train_loader): 2
len(val_loader)  : 1

=== one batch shapes ===
ch : (32, 16, 2048)  -> (B,T,F_in)
y  : (32, 2048)  -> (B,F_out)
yhat: (32, 2048)  -> (B,F_out)
this forward predicted vectors: 32 (=B)
each vector predicts elements: 2048 (=F_out)


: 